In [1]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

Looking in indexes: https://download.pytorch.org/whl/cu124
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Users\avina\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import torch
torch.__version__

'2.6.0+cu124'

In [3]:
torch.cuda.is_available()

True

In [4]:
# creating pyTorch tensor
import torch
tensor0d=torch.tensor(4)
tensor1d=torch.tensor([1,2,3])
tensor2d=torch.tensor([[1,2,3],[4,5,6]])


In [5]:
tensor1d.dtype

torch.int64

In [6]:
float_tensor=torch.tensor([1.0,2.0,3.0])
float_tensor.dtype

torch.float32

In [7]:
float_tensor=tensor1d.type(torch.float32)
float_tensor.dtype

torch.float32

In [8]:
# common tensor operations
print(tensor2d.shape)

torch.Size([2, 3])


In [9]:
print(tensor2d.reshape(3,2))

tensor([[1, 2],
        [3, 4],
        [5, 6]])


In [10]:
print(tensor2d.view(3,2))

tensor([[1, 2],
        [3, 4],
        [5, 6]])


In [11]:
# transpose not same as reshape or view
print(tensor2d.T)

tensor([[1, 4],
        [2, 5],
        [3, 6]])


In [12]:
# tensor multiplication 
print(tensor2d @ tensor2d.T)

tensor([[14, 32],
        [32, 77]])


In [13]:
import torch.nn.functional as F  # functional API for loss functions

# target label (ground truth)
y = torch.tensor([1.0])

# input feature and model parameters
x1 = torch.tensor([1.1])       # input value
w1 = torch.tensor([2.2])       # weight
b = torch.tensor([0.0])        # bias

# linear combination: z = w*x + b
z = w1 * x1 + b

# apply sigmoid to get probability
a = torch.sigmoid(z)

# binary cross-entropy loss between prediction and target
loss = F.binary_cross_entropy(a, y)

# print final loss value
print(loss)

tensor(0.0852)


In [14]:
# computing gradients via autograd
import torch.nn.functional as F
from torch.autograd import grad

y= torch.tensor([1.0])
x1=torch.tensor([1.1])
w1=torch.tensor([2.2], requires_grad=True)
b=torch.tensor([0.0], requires_grad=True)

z = w1 * x1 + b
a = torch.sigmoid(z)
loss = F.binary_cross_entropy(a, y)

# compute gradients of loss w.r.t. parameters
grad_L_w1 = grad(loss, w1, retain_graph=True)
grad_L_b = grad(loss, b, retain_graph=True)

print(grad_L_w1)
print(grad_L_b)

(tensor([-0.0898]),)
(tensor([-0.0817]),)


In [15]:
loss.backward()
print(w1.grad)
print(b.grad)

tensor([-0.0898])
tensor([-0.0817])


In [16]:
# a multilayer perceptron with two hidden layers
class NeuralNetwork(torch.nn.Module):
    def __init__(self,num_inputs,num_outputs):
        super().__init__()

        self.layers = torch.nn.Sequential(
            # input layer to hidden layer 1
            torch.nn.Linear(num_inputs,30),
            torch.nn.ReLU(),
            # hidden layer 1 to hidden layer 2
            torch.nn.Linear(30,20),
            torch.nn.ReLU(),
            # hidden layer 2 to output layer
            torch.nn.Linear(20,num_outputs)
        )
    def forward(self,x):
        logits = self.layers(x)
        return logits
    

In [17]:
model=NeuralNetwork(num_inputs=50,num_outputs=3)

In [18]:
model

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)

In [19]:
# total number of parameters
num_params=sum(p.numel() for p in model.parameters() if p.requires_grad)

In [20]:
num_params

2213

In [21]:
model.layers[0].weight

Parameter containing:
tensor([[-0.1032,  0.0519,  0.0286,  ..., -0.0140,  0.0634, -0.1358],
        [-0.1154,  0.0364, -0.0494,  ..., -0.0737, -0.0520,  0.0473],
        [-0.0363, -0.1045, -0.1139,  ..., -0.0476,  0.0543,  0.1038],
        ...,
        [ 0.0398,  0.0763, -0.0437,  ...,  0.0797,  0.0231, -0.1405],
        [ 0.1241, -0.1298,  0.1289,  ...,  0.0750, -0.0710, -0.0264],
        [-0.0777, -0.0686,  0.0364,  ..., -0.1132,  0.0917, -0.0845]],
       requires_grad=True)

In [22]:
model.layers[0].weight.shape

torch.Size([30, 50])

In [23]:
# reproduceable weights
torch.manual_seed(123)
model=NeuralNetwork(num_inputs=50,num_outputs=3)


In [24]:
model.layers[0].weight

Parameter containing:
tensor([[-0.0577,  0.0047, -0.0702,  ...,  0.0222,  0.1260,  0.0865],
        [ 0.0502,  0.0307,  0.0333,  ...,  0.0951,  0.1134, -0.0297],
        [ 0.1077, -0.1108,  0.0122,  ...,  0.0108, -0.1049, -0.1063],
        ...,
        [-0.0787,  0.1259,  0.0803,  ...,  0.1218,  0.1303, -0.1351],
        [ 0.1359,  0.0175, -0.0673,  ...,  0.0674,  0.0676,  0.1058],
        [ 0.0790,  0.1343, -0.0293,  ...,  0.0344, -0.0971, -0.0509]],
       requires_grad=True)

In [25]:
# inspecting forward pass
torch.manual_seed(123)
X=torch.rand((1,50))



In [26]:
X

tensor([[0.2961, 0.5166, 0.2517, 0.6886, 0.0740, 0.8665, 0.1366, 0.1025, 0.1841,
         0.7264, 0.3153, 0.6871, 0.0756, 0.1966, 0.3164, 0.4017, 0.1186, 0.8274,
         0.3821, 0.6605, 0.8536, 0.5932, 0.6367, 0.9826, 0.2745, 0.6584, 0.2775,
         0.8573, 0.8993, 0.0390, 0.9268, 0.7388, 0.7179, 0.7058, 0.9156, 0.4340,
         0.0772, 0.3565, 0.1479, 0.5331, 0.4066, 0.2318, 0.4545, 0.9737, 0.4606,
         0.5159, 0.4220, 0.5786, 0.9455, 0.8057]])

In [27]:
out=model(X)

In [28]:
print(out)

tensor([[-0.1262,  0.1080, -0.1792]], grad_fn=<AddmmBackward0>)


In [29]:
# if we are using models for prediction only we can turn off gradients to save memory and computations
with torch.no_grad():
    out=model(X)
print(out)

tensor([[-0.1262,  0.1080, -0.1792]])


In [30]:
# using softmax to get probabilities from logits
with torch.no_grad():
    out=model(X)
    probs=F.softmax(out,dim=1)
probs

tensor([[0.3113, 0.3934, 0.2952]])

In [32]:
# Setting up efficient data loaders
X_train=torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, -1.5]
])
y_train=torch.tensor([0,0,0,1,1])
X_test=torch.tensor([
    [-0.8, 2.8],
    [2.6,-1.6]
])

y_test=torch.tensor([0,1])

In [43]:
#defining a dataset class
from torch.utils.data import Dataset
class ToyDataset(Dataset):
    def __init__(self,X,y):
        self.features=X
        self.labels=y
    def __getitem__(self,idx):
        one_x=self.features[idx]
        one_y=self.labels[idx]
        return one_x,one_y
    def __len__(self):
        return self.labels.shape[1]


In [34]:
train_ds=ToyDataset(X_train,y_train)
test_ds=ToyDataset(X_test,y_test)


In [44]:
len(train_ds)

5

In [45]:
# using dataloader
from torch.utils.data import DataLoader
torch.manual_seed(123)

train_loader=DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0
)

test_loader=DataLoader(
    dataset=test_ds,
    batch_size=2,
    shuffle=False,
    num_workers=0
)


In [46]:
# iterating through the dataloader
for idx, (x,y) in enumerate(train_loader):
    print(f"Batch {idx+1}:", x, y)

Batch 1: tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) tensor([1, 0])
Batch 2: tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])
Batch 3: tensor([[ 2.7000, -1.5000]]) tensor([1])


 In practice, having a substantially smaller batch as the last batch in a training epoch can disturb the convergence during training. To prevent this, set drop_last=True

In [47]:
train_loader=DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0,
    drop_last=True
)


In [48]:
for idx, (x,y) in enumerate(train_loader):
    print(f"Batch {idx+1}:", x, y)

Batch 1: tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])
Batch 2: tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) tensor([1, 0])


In [56]:
# training our neural network
import torch.nn.functional as F

torch.manual_seed(123)
model=NeuralNetwork(num_inputs=2,num_outputs=2)
optimizer=torch.optim.SGD(model.parameters(),lr=0.5)

num_epochs=3
for epoch in range(num_epochs):
    model.train()
    for batch_idx,(features,labels) in enumerate(train_loader):
        logits=model(features)
        loss=F.cross_entropy(logits,labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # logging
        print(f"Epoch {epoch+1:03d}/{num_epochs:03d}"
              f" | Batch {batch_idx+1:03d}/{len(train_loader):03d}"
              f" | Loss: {loss.item():.2f}")
# evaluating our model
model.eval()
with torch.no_grad():
    outputs=model(X_train)
print("Train outputs:", outputs)
    

Epoch 001/003 | Batch 001/002 | Loss: 0.75
Epoch 001/003 | Batch 002/002 | Loss: 0.65
Epoch 002/003 | Batch 001/002 | Loss: 0.44
Epoch 002/003 | Batch 002/002 | Loss: 0.13
Epoch 003/003 | Batch 001/002 | Loss: 0.03
Epoch 003/003 | Batch 002/002 | Loss: 0.00
Train outputs: tensor([[ 2.8569, -4.1618],
        [ 2.5382, -3.7548],
        [ 2.0944, -3.1820],
        [-1.4814,  1.4816],
        [-1.7176,  1.7342]])


In [57]:
# obtaining class memberships using softmax
torch.set_printoptions(sci_mode=False)
probas=torch.softmax(outputs,dim=1)
print(probas)

tensor([[    0.9991,     0.0009],
        [    0.9982,     0.0018],
        [    0.9949,     0.0051],
        [    0.0491,     0.9509],
        [    0.0307,     0.9693]])


In [59]:
# printing predicted class labels
pred_labels=torch.argmax(probas,dim=1)
print(pred_labels)

tensor([0, 0, 0, 1, 1])


In [61]:
# we can also get predicted class labels directly from the logits using argmax
pred_labels_direct=torch.argmax(outputs,dim=1)
print(pred_labels_direct)

tensor([0, 0, 0, 1, 1])


In [62]:
torch.sum(pred_labels==y_train)

tensor(5)

In [ ]:
def accuracy(model, dataloader):
    model=model.eval()
    correct=0
    total_examples=0

    for idx, (features,labels) in enumerate(dataloader):
        